<a href="https://colab.research.google.com/github/jjmin2/CV_Project_9team/blob/main/CV_Evaluation_Comparison_of_AUROC_and_IoU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import sys

sys.path.append('/content/drive/MyDrive/CV/activeLearning')
sys.path.append('/content/drive/MyDrive/CV/activeLearning/sobel_detector.py')
sys.path.append('/content/drive/MyDrive/CV/activeLearning/padim.py')

In [3]:
import os
import glob
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

# 1. 데이터셋 클래스 선언
class MVTecDataset(Dataset):
    def __init__(self, data_dir, category, split='train'):
        self.data_dir = os.path.join(data_dir, category, split)
        self.split = split
        self.image_paths = sorted(glob.glob(os.path.join(self.data_dir, '**', '*.png'), recursive=True))

        self.transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor()
        ])
        self.mask_transform = transforms.Compose([
            transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        img_tensor = self.transform(image)

        if self.split == 'train':
            label = 0
            mask_tensor = torch.zeros((1, 256, 256))
        else:
            defect_type = os.path.basename(os.path.dirname(img_path))
            if defect_type == 'good':
                label = 0
                mask_tensor = torch.zeros((1, 256, 256))
            else:
                label = 1
                mask_path = img_path.replace('test', 'ground_truth').replace('.png', '_mask.png')
                if os.path.exists(mask_path):
                    mask_tensor = self.mask_transform(Image.open(mask_path).convert('L'))
                else:
                    mask_tensor = torch.zeros((1, 256, 256))

        return img_tensor, torch.tensor(label, dtype=torch.float32), mask_tensor, img_path

DATA_DIR = '/content/drive/MyDrive/CV/activeLearning/mvtec_anomaly_detection'
CATEGORY = 'bottle'

dataset_train = MVTecDataset(DATA_DIR, CATEGORY, split='train')
dataset_val = MVTecDataset(DATA_DIR, CATEGORY, split='val')
dataset_test = MVTecDataset(DATA_DIR, CATEGORY, split='test')

In [4]:
import time
import numpy as np
import pandas as pd
import cv2
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score

# ==========================================
# 0. 평가지표 및 헬퍼 함수
# ==========================================
def calculate_auroc(y_true, y_scores):
    return roc_auc_score(y_true, y_scores)

def calculate_pixel_auroc(mask_true, score_map):
    return roc_auc_score(mask_true.flatten(), score_map.flatten())

def calculate_iou(mask_true, mask_pred):
    intersection = np.logical_and(mask_true, mask_pred).sum()
    union = np.logical_or(mask_true, mask_pred).sum()
    return np.nan if union == 0 else intersection / union

def tensor_to_bgr(tensor):
    # PaDiM 입력 규격 (BGR uint8) 변환
    img_rgb = (tensor.squeeze(0).permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)

from sobel_detector import SobelDetector
from padim import PaDiMDetector

# ==========================================
# 1. 모델 초기화 및 정상 기준 세팅
# ==========================================
print("1. Sobel 및 PaDiM 모델 세팅 시작...")

# 테스트 데이터셋에서 정상(라벨 0) 이미지들만 추출하여 검증용으로 사용
val_images = [dataset_test[i][0] for i in range(len(dataset_test)) if dataset_test[i][1].item() == 0]
print(f"검증용(val) 정상 이미지 {len(val_images)}장 추출 완료.")

# --- Sobel 세팅 ---
sobel_model = SobelDetector()
train_images = [dataset_train[i][0] for i in range(len(dataset_train))]
sobel_model.fit(train_images)
sobel_model.calibrate(val_images) # 비어있지 않은 val_images 전달
print("- [Sobel] 정상 통계 추정 및 임계값 보정 완료")

# --- PaDiM 세팅 ---
padim_model = PaDiMDetector()
train_pt_path = '/content/drive/MyDrive/bottle_train_features.pt'
padim_model.fit_from_file(train_pt_path) # 정상 특징 및 인덱스 로드

val_score_maps_padim = []
for img_tensor in val_images:
    # Sobel과 동일하게 추출해둔 검증용 정상 이미지를 평가하여 임계값 산출
    val_score_maps_padim.append(padim_model.score_map(tensor_to_bgr(img_tensor)))

padim_pixel_threshold = np.percentile(np.stack(val_score_maps_padim), 99.5)
print(f"- [PaDiM] 정상 통계 로드 및 픽셀 임계값({padim_pixel_threshold:.4f}) 보정 완료\n")

# ==========================================
# 2. 테스트 데이터 통합 평가 진행
# ==========================================
print("2. 테스트 데이터 통합 평가 진행 중...")
dataloader_test = DataLoader(dataset_test, batch_size=1, shuffle=False)

# 결과 저장용 딕셔너리
results = {
    'Sobel': {'y_true': [], 'y_scores': [], 'pixel_auroc': [], 'iou': [], 'time': []},
    'PaDiM': {'y_true': [], 'y_scores': [], 'pixel_auroc': [], 'iou': [], 'time': []}
}

for img_tensor, label, mask_tensor, _ in dataloader_test:
    single_img = img_tensor.squeeze(0)
    img_bgr = tensor_to_bgr(img_tensor)
    mask_numpy = mask_tensor.squeeze().numpy()
    lbl = label.item()

    results['Sobel']['y_true'].append(lbl)
    results['PaDiM']['y_true'].append(lbl)

    # --- Sobel 예측 ---
    sobel_res = sobel_model.predict(single_img)
    results['Sobel']['time'].append(sobel_res["elapsed_ms"])
    results['Sobel']['y_scores'].append(sobel_res["image_score"])

    # --- PaDiM 예측 ---
    start_time = time.perf_counter()
    padim_score_map = padim_model.score_map(img_bgr)
    results['PaDiM']['time'].append((time.perf_counter() - start_time) * 1000)
    results['PaDiM']['y_scores'].append(float(np.max(padim_score_map)))

    # --- 픽셀 단위 평가 (정답 마스크가 있을 경우) ---
    if len(np.unique(mask_numpy)) > 1:
        # Pixel AUROC
        results['Sobel']['pixel_auroc'].append(calculate_pixel_auroc(mask_numpy, sobel_res["score_map"]))
        results['PaDiM']['pixel_auroc'].append(calculate_pixel_auroc(mask_numpy, padim_score_map))

    if lbl == 1:
        # IoU 계산 (불량 이미지 한정)
        sobel_iou = calculate_iou(mask_numpy, sobel_res["mask"])
        if not np.isnan(sobel_iou):
            results['Sobel']['iou'].append(sobel_iou)

        padim_pred_mask = (padim_score_map > padim_pixel_threshold).astype(np.uint8)
        padim_iou = calculate_iou(mask_numpy, padim_pred_mask)
        if not np.isnan(padim_iou):
            results['PaDiM']['iou'].append(padim_iou)

# ==========================================
# 3. 최종 지표 산출 및 통합 CSV 저장
# ==========================================
final_data = []
print("-" * 50)
for model_name in ['Sobel', 'PaDiM']:
    r = results[model_name]
    img_auroc = calculate_auroc(r['y_true'], r['y_scores'])
    px_auroc = np.mean(r['pixel_auroc']) if r['pixel_auroc'] else 0
    mean_iou = np.mean(r['iou']) if r['iou'] else 0
    mean_time = np.mean(r['time'])
    med_time = np.median(r['time'])

    print(f"[{model_name} 최종 결과]")
    print(f"Image AUROC: {img_auroc:.4f}")
    print(f"Pixel AUROC: {px_auroc:.4f}")
    print(f"Mean IoU   : {mean_iou:.4f}")
    print(f"Process Time: {mean_time:.2f} ms (Mean) / {med_time:.2f} ms (Median)\n")

    final_data.append({
        'Method': model_name,
        'Image AUROC': img_auroc,
        'Pixel AUROC (Mean)': px_auroc,
        'Mean IoU (Defective only)': mean_iou,
        'Processing Time Mean (ms)': mean_time,
        'Processing Time Median (ms)': med_time
    })

df_combined = pd.DataFrame(final_data)
df_combined.to_csv('/content/drive/MyDrive/evaluation_results_combined.csv', index=False)
print("evaluation_results_combined.csv 통합 저장 완료")

1. Sobel 및 PaDiM 모델 세팅 시작...
검증용(val) 정상 이미지 20장 추출 완료.
- [Sobel] 정상 통계 추정 및 임계값 보정 완료
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:01<00:00, 38.5MB/s]


- [PaDiM] 정상 통계 로드 및 픽셀 임계값(9.9005) 보정 완료

2. 테스트 데이터 통합 평가 진행 중...
--------------------------------------------------
[Sobel 최종 결과]
Image AUROC: 0.9810
Pixel AUROC: 0.7947
Mean IoU   : 0.2030
Process Time: 4.00 ms (Mean) / 3.00 ms (Median)

[PaDiM 최종 결과]
Image AUROC: 0.9873
Pixel AUROC: 0.9818
Mean IoU   : 0.4329
Process Time: 131.83 ms (Mean) / 100.27 ms (Median)

evaluation_results_combined.csv 통합 저장 완료
